## Gerar bases utilizando Databricks Labs Data Generator
- [Databricks Labs Data Generator](https://databrickslabs.github.io/dbldatagen/public_docs/index.html) 

In [0]:
%run ../common/setenv

In [0]:
!pip install dbldatagen

In [0]:
import os
import dbldatagen as dg
import pandas as pd

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType

### Gerar base de Clientes
- Gerar 50000 clients
- Gravar cvs no volume files/generate
- link exemplo: https://databrickslabs.github.io/dbldatagen/public_docs/multi_table_data.html#let-s-model-our-customers

- Estes arquivos serão usados para os demais laboratórios 

In [0]:
partitions_requested = 8
spark.conf.set("spark.sql.shuffle.partitions", partitions_requested)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 20000)

customer_ids = 50000
customer_min_id = 1000

spark.catalog.clearCache()  # clear cache so that if we run multiple times to check
                            # performance, we're not relying on cache

data_rows = customer_ids

customer_dataspec = (dg.DataGenerator(spark, rows=data_rows, partitions=partitions_requested)
            .withColumn("customer_id", DecimalType(10), minValue=customer_min_id, uniqueValues=customer_ids)
            .withColumn("customer_name", StringType(), template=r"\\w \\w|\\w a. \\w")
            .withColumn("phone_number", DecimalType(10),  minValue=1000000000,
                        baseColumn=["customer_id", "customer_name"], baseColumnType="hash")
            # for email, we'll just use the formatted phone number
            .withColumn("email", StringType(), format="subscriber_%s@myoperator.com",
                        baseColumn="phone_number")
            )

df_customers = (customer_dataspec.build()
                .dropDuplicates(["phone_number"])
                .orderBy("customer_id")
                .cache()
               )

print(f"revised customers    : {df_customers.count()}")
print(f"unique customers     : {df_customers.select(F.countDistinct('customer_id')).take(1)[0][0]}")
print(f"unique phone numbers : {df_customers.select(F.countDistinct('phone_number')).take(1)[0][0]}")

# display(df_customers)

In [0]:
# Gravar dataframe em "n" arquivos csv

df_customers_pd = df_customers.toPandas()

output_path = f"{settings['lab.path.files']}/generated"
os.makedirs(output_path, exist_ok=True)

file_name = f"{output_path}/customers.csv"
df_customers_pd.to_csv(file_name, index=False)

### Gerar base Vendas
- Gerar 1.000.000 vendas, entre 1/1/2025 e 28/2/2025
- Gravar as vendas em arquivos cvs, um arquivo por data

In [0]:
# https://databrickslabs.github.io/dbldatagen/public_docs/DATARANGES.html#examples

row_count = 1000000

sales_dataspec = (
    dg.DataGenerator(spark, 
                     name="test_data_set1", 
                     rows=row_count, partitions=4, 
                     randomSeedMethod="hash_fieldname", verbose=True, )
    .withColumn("customer_id", IntegerType(), minValue=customer_min_id, maxValue=(customer_min_id+customer_ids), random=True)
    .withColumn("purchase_id", IntegerType(), minValue=1000000, maxValue=2000000)
    .withColumn("product_code", IntegerType(), uniqueValues=1000, random=True)
    .withColumn("purchase_amount", IntegerType(), minValue=1, maxValue=100, random=True)
    .withColumn("purchase_unit_value", DecimalType(10,2), minValue=1, maxValue=3000, random=True)
    .withColumn(
        "purchase_date",
        DateType(),
        data_range=dg.DateRange("2025-01-01 00:00:00", "2025-02-28 00:00:00","days=1"),
        random=True,
    )
)

df_sales = sales_dataspec.build()
# display(df_sales)

In [0]:
# Save into csv
output_path = f"{settings['lab.path.files']}/generated/sales"
os.makedirs(output_path, exist_ok=True)

# Convert Spark DataFrame to Pandas DataFrame
df_sales_pd = df_sales.toPandas()

# Group by purchase_date and save each group to a separate CSV file
for date, group in df_sales_pd.groupby('purchase_date'):
    file_name = f"{output_path}/sales_{date.strftime('%Y%m%d')}.csv"
    group.to_csv(file_name, index=False)